# Week 3, Notebook 5: Machine Learning for Finance

**Teaching a computer to spot patterns in money.**

*Author: The Genius Project Year 3*

---

### What you will build
- **Part A:** an ML model that predicts *next month's spending* from your habits.
- **Part B:** an ML model that predicts whether the stock goes *up or down* tomorrow.
- The most important skill of all: telling a real edge apart from wishful thinking.

### The big idea
Machine learning is pattern-spotting at scale: you hand a model many examples, each
described by **features**, and it learns which features point to which outcome. In
Week 2 you used these exact tools, logistic regression and decision trees,
on football. Here you point them at money.

## Part A: Predict next month's spending (regression)

**Regression** predicts a number. Our number is next month's spending. First we build
**features** (clues the model can learn from) out of the budget history.
Good features are the heart of ML.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

budget = pd.read_csv("data/monthly_budget.csv", parse_dates=["month"])

# Feature engineering: describe each month with clues known BEFORE it happens.
budget["last_month_spending"] = budget["spending"].shift(1)   # what we spent last month
budget["avg_last_3"] = budget["spending"].shift(1).rolling(3).mean()  # recent trend
budget["month_num"] = budget["month"].dt.month                # calendar month (seasonality)

feats = ["last_month_spending", "avg_last_3", "income", "month_num"]
data = budget.dropna(subset=feats + ["spending"]).reset_index(drop=True)

X = data[feats]
y = data["spending"]   # the target we want to predict
data[["month", "spending"] + feats].head()

We split by time again (train on the earlier months, test on the most recent 12) and
compare two models against the naive "same as last month" baseline.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

TEST = 12
X_train, X_test = X.iloc[:-TEST], X.iloc[-TEST:]
y_train, y_test = y.iloc[:-TEST], y.iloc[-TEST:]


def mae(a, p):
    return float(np.mean(np.abs(np.array(a) - np.array(p))))


# Baseline: guess this month = last month.
baseline_pred = X_test["last_month_spending"]

linreg = LinearRegression().fit(X_train, y_train)
tree = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, y_train)

print(f"Baseline (last month)   MAE ${mae(y_test, baseline_pred):.2f}")
print(f"Linear regression       MAE ${mae(y_test, linreg.predict(X_test)):.2f}")
print(f"Decision tree           MAE ${mae(y_test, tree.predict(X_test)):.2f}")

### Which clues did the linear model lean on?

A linear model gives each feature a **weight**. A positive weight pushes the
prediction up, a negative weight pulls it down, and a bigger size means the feature
mattered more, just like the logic model in Week 2.

In [ ]:
weights = pd.Series(linreg.coef_, index=feats).sort_values()
plt.figure(figsize=(8, 3.5))
colors = ["#C44E52" if w < 0 else "#55A868" for w in weights]
plt.barh(weights.index, weights.values, color=colors)
plt.title("What drives next month's spending?")
plt.xlabel("Weight (effect on the prediction)")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

## Part B: Predict the stock's direction (classification)

**Classification** predicts a category, not a number. Our category is simple:
does the price go **up** (1) or **down** (0) tomorrow? This is the harder side of
ML for markets.

In [ ]:
stock = pd.read_csv("data/daily_stock.csv", parse_dates=["date"]).set_index("date")
stock["return"] = stock["price"].pct_change()

# Features: recent behaviour the model can see today.
stock["ret_1"] = stock["return"].shift(1)              # yesterday's move
stock["ret_2"] = stock["return"].shift(2)              # the day before
stock["avg_5"] = stock["return"].shift(1).rolling(5).mean()   # recent momentum
stock["wobble_5"] = stock["return"].shift(1).rolling(5).std()  # recent volatility
stock["vol_change"] = stock["volume"].pct_change().shift(1)    # trading activity

# Target: did tomorrow go up? (1 = up, 0 = down)
stock["up_tomorrow"] = (stock["return"].shift(-1) > 0).astype(int)

feat_cols = ["ret_1", "ret_2", "avg_5", "wobble_5", "vol_change"]
d = stock.dropna(subset=feat_cols + ["up_tomorrow"])
print("Usable days:", len(d))
print(f"Days the stock actually rose: {d['up_tomorrow'].mean() * 100:.1f}%")

The line above is our **baseline to beat**: if the stock rises on, say, 52% of days,
a lazy model that always guesses "up" is already 52% accurate. A real model has to do
**better than that** to be worth anything.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

Xc = d[feat_cols]
yc = d["up_tomorrow"]

TEST = int(len(d) * 0.2)                 # last 20% of days, kept hidden
Xc_train, Xc_test = Xc.iloc[:-TEST], Xc.iloc[-TEST:]
yc_train, yc_test = yc.iloc[:-TEST], yc.iloc[-TEST:]

# Baseline: always guess the more common outcome from the training days.
always_guess = int(yc_train.mean() >= 0.5)
baseline_acc = accuracy_score(yc_test, [always_guess] * len(yc_test))

models = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "Decision tree": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42),
}

print(f"Baseline (always guess {'up' if always_guess else 'down'}): {baseline_acc * 100:.1f}%")
for name, m in models.items():
    m.fit(Xc_train, yc_train)
    acc = accuracy_score(yc_test, m.predict(Xc_test))
    print(f"{name:<22} {acc * 100:.1f}%")

### The honest takeaway

Your models land right around the baseline, a coin-flip's distance from
guessing. That is **not** a failure of your code; it is the truth about markets. Back
in Notebook 3 the returns had almost **no memory** (autocorrelation near zero), so
there is barely any pattern for the model to grab. Thousands of professionals hunt for
tiny edges here, which is exactly why they are tiny.

**The real skill of ML is not getting a high number, it is knowing whether the
number means anything.** Part A worked because spending is habit-driven and
predictable. Part B struggled because prices are close to random. Same tools, very
different problems.

> **Watch out for overfitting.** Give the decision tree `max_depth=None` and its
> training accuracy will look amazing while its test accuracy stays near the coin
> flip. A model that memorises the past has learned nothing about the future. Always
> judge on the hidden test set.

### What you learned
- **Regression** predicts a number; **classification** predicts a category.
- **Feature engineering** (building good clues) matters more than the model.
- Always beat a **baseline**, or your accuracy is meaningless.
- Predictable problems (spending) and near-random ones (prices) need the same tools
  but give very different results.

### Try it yourself
- Add a feature for the stock, like a 10-day average return, and see if accuracy moves.
- In Part A, print `tree.feature_importances_` to see which clue the tree trusted most.
- Set the tree's `max_depth=None` and compare training vs test accuracy. See overfitting live.

### Next
The bonus notebook trades the straight line for a small **neural network**, the
same family of models powering the AI you hear about, and asks whether more
brainpower cracks a near-random market.